# Optional memory strategies

Memory is independent of the PTC tool.

| Strategy | Durable source | Model-visible retrieval | Compaction |
| --- | --- | --- | --- |
| disabled | Operational task/session stores | None | Existing bounded work packets |
| `trace_native` | Canonical append-only ledger | Versioned, bounded `memory` programs | Evidence-addressed deterministic views |
| `pi` | ADK session history | None | Pi-derived structured summary with a recent raw tail |

The Pi summary is advisory. Trace-native views carry evidence identities and can support recovery contracts.


In [ ]:
import hashlib
import json

EVENTS = [
    {"seq": 1, "kind": "task.started", "text": "Fix parser timeout"},
    {"seq": 2, "kind": "file.read", "path": "parser.py", "text": "recursive branch"},
    {"seq": 3, "kind": "test.run", "status": "timeout", "text": "timed out after 30s"},
    {"seq": 4, "kind": "file.edit", "path": "parser.py", "status": "completed"},
]

def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))

assert [event["seq"] for event in EVENTS] == [1, 2, 3, 4]


## Trace-native: a versioned program over evidence

The result identity includes the program version, watermark, parameters, canonical result bytes, and evidence sequence numbers. Repeating the same query at the same watermark returns identical bytes.


In [ ]:
def trace_progress(events, *, version=1, watermark=None):
    selected = [row for row in events if watermark is None or row["seq"] <= watermark]
    data = {
        "completed_edits": [row["path"] for row in selected if row["kind"] == "file.edit" and row.get("status") == "completed"],
        "failures": [row["text"] for row in selected if row.get("status") == "timeout"],
    }
    envelope = {
        "program": "task.progress",
        "version": version,
        "watermark": selected[-1]["seq"] if selected else 0,
        "evidence_seqs": [row["seq"] for row in selected],
        "data": data,
    }
    envelope["content_hash"] = hashlib.sha256(canonical(data).encode()).hexdigest()
    return envelope

first = trace_progress(EVENTS)
second = trace_progress(EVENTS)
assert canonical(first) == canonical(second)
assert first["data"]["completed_edits"] == ["parser.py"]
print(canonical(first))


## Pi: structured checkpoint plus recent raw tail

Pi compacts when context exceeds `context_window - reserve_tokens` (default reserve: 16,384). Older messages become a structured continuation summary; recent messages stay exact. Skein delegates the event-range persistence to ADK and ports Pi's summary headings.


In [ ]:
PI_HEADINGS = (
    "Goal", "Constraints & Preferences", "Progress", "Key Decisions", "Next Steps", "Critical Context"
)

def should_compact(context_tokens, context_window=200_000, reserve_tokens=16_384):
    return context_tokens > context_window - reserve_tokens


def pi_checkpoint(events, keep_recent=2):
    older, tail = events[:-keep_recent], events[-keep_recent:]
    summary = {
        "Goal": older[0]["text"] if older else "(none)",
        "Constraints & Preferences": [],
        "Progress": {"Done": [], "In Progress": [row["kind"] for row in tail], "Blocked": []},
        "Key Decisions": [],
        "Next Steps": [f"Continue after event {tail[-1]['seq']}"] if tail else [],
        "Critical Context": [row.get("text", row["kind"]) for row in older[1:]],
    }
    return {"summary": summary, "retained_tail": tail}

assert not should_compact(180_000)
assert should_compact(190_000)
checkpoint = pi_checkpoint(EVENTS)
assert tuple(checkpoint["summary"]) == PI_HEADINGS
assert [row["seq"] for row in checkpoint["retained_tail"]] == [3, 4]
print(canonical(checkpoint))


## What the comparison isolates

Set `memory.enabled: false` for no optional memory, or enable it with `implementation: trace_native` or `pi`. Trace-native alone permits context programs, prior-run recall, working notes, conversation-notebook continuity, and safe-auto recovery. Hold the PTC implementation fixed while comparing memory strategies.
